In [58]:
import pandas as pd

# Example DataFrame setup
data = {
    'symbol': ['AAPL', 'AAPL', 'ABBV', 'ABBV', 'AAPL', 'ABBV'],
    'close': [190, 195, 145, 150, 200, 155],
    'date': ['2023-08-01', '2023-08-15', '2023-08-01', '2023-08-15', '2023-09-01', '2023-09-05']
}
df = pd.DataFrame(data)

# Convert 'date' to datetime format
df['date'] = pd.to_datetime(df['date'])

# Sort by 'symbol' and 'date'
df = df.sort_values(by=['symbol', 'date'])

# Create a 'year_month' column
df['year_month'] = df['date'].dt.to_period('M')

# Calculate the first available price for each symbol at the start of each month
monthly_first_available = df.groupby(['symbol', 'year_month']).first().reset_index()

# Rename the 'close' column in monthly_first_available
monthly_first_available = monthly_first_available.rename(columns={'close': 'first_available_price'})

# Merge the calculated first available prices back into the original DataFrame
df = df.merge(
    monthly_first_available[['symbol', 'year_month', 'first_available_price']],
    on=['symbol', 'year_month'],
    how='left'
)

# Drop duplicates to ensure only one entry per month
df = df.drop_duplicates(subset=['symbol', 'year_month'])

# Clean up
df = df.drop('year_month', axis=1)

# Set 'date' as index
df = df.set_index('date')

# Display the results
print(df[['symbol', 'close', 'first_available_price']].head(10))

           symbol  close  first_available_price
date                                           
2023-08-01   AAPL    190                    190
2023-09-01   AAPL    200                    200
2023-08-01   ABBV    145                    145
2023-09-05   ABBV    155                    155


In [59]:
help(df.groupby)

Help on method groupby in module pandas.core.frame:

groupby(by=None, axis: 'Axis | lib.NoDefault' = <no_default>, level: 'IndexLabel | None' = None, as_index: 'bool' = True, sort: 'bool' = True, group_keys: 'bool' = True, observed: 'bool | lib.NoDefault' = <no_default>, dropna: 'bool' = True) -> 'DataFrameGroupBy' method of pandas.core.frame.DataFrame instance
    Group DataFrame using a mapper or by a Series of columns.
    
    A groupby operation involves some combination of splitting the
    object, applying a function, and combining the results. This can be
    used to group large amounts of data and compute operations on these
    groups.
    
    Parameters
    ----------
    by : mapping, function, label, pd.Grouper or list of such
        Used to determine the groups for the groupby.
        If ``by`` is a function, it's called on each value of the object's
        index. If a dict or Series is passed, the Series or dict VALUES
        will be used to determine the groups (th